**Author**: JW Debelius (justine.debelius@jhu.edu)<br>
**Project**: 017-Arg<br>
**Enviroment**: <br>
**Creation Date**: July 2021<br>
**Update Date**: November 2025

This notebbok will take the metadata file and sync with the feature tables.

In [1]:
import os
import shutil

import biom
import numpy as np
import pandas as pd
import scipy
import skbio

from qiime2 import Artifact, Metadata, Visualization

import qiime2.plugins.boots.actions as q2_boots
import qiime2.plugins.diversity.actions as q2_diversity
import qiime2.plugins.feature_table.actions as q2_feature_table
import qiime2.plugins.gemelli.actions as q2_gemelli
import qiime2.plugins.longitudinal.actions as q2_longitudinal
import qiime2.plugins.taxa.actions as q2_taxa

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_demux/_summarize/_visualizer.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Sample metadata

We define opur analytical metdata as 

In [3]:
meta = pd.read_csv('final-data/01.metadata/analysis_sample_meta.tsv',
                   sep='\t',
                   dtype=str)
meta.set_index('sample-id', inplace=True)

In [4]:
meta.columns

Index(['host_subject_id', 'age', 'sex', 'antibiotic', 'spectrum',
       'days_since_intervention', 'collection_date', 'intervention_date',
       'timepoint', 'tidy_num', 'omeprazol_freq', 'probiotic_frequency',
       'vegetarian', 'after_travel_na_eu', 'antibiotics_last_12_mo',
       'days_since_last_abx', 'project_name', 'extraction_no',
       'extraction_well_pos', 'sequencer_id', 'flowcell_id', 'geo_loc_name',
       'abx_code2', 'gtdb_depth', 'size_frac', 'alt', 'source_mat_id',
       'samp_collect_device', 'env_broad_scale', 'env_local_scale',
       'env_material', 'lat_lon', 'elev', 'host_age', 'host_sex',
       'replace_control'],
      dtype='object')

In [5]:
working_meta = meta.copy()
float_cols = ['age', 
              'days_since_intervention',
              'timepoint',
              'gtdb_depth']
working_meta[float_cols] = working_meta[float_cols].astype(float)
working_meta_q2 = Metadata(working_meta)

And then we'll read in the taxonomy tables and import everything associated into QIIME 2.

In [6]:
species_table = biom.load_table('data/02.tables/gtdb_species.biom')
genome_table = biom.load_table('data/02.tables/gtdb_genome.biom')

In [7]:
species_q2 = Artifact.import_data('FeatureTable[Frequency]', species_table, biom.Table)
genome_q2 = Artifact.import_data('FeatureTable[Frequency]', genome_table, biom.Table)

In [8]:
species_m_q2, = q2_feature_table.filter_samples(species_q2, metadata=Metadata(meta))
genome_m_q2, = q2_feature_table.filter_samples(genome_q2, metadata=Metadata(meta))

In [9]:
genome_taxa = genome_table.metadata_to_dataframe(axis='observation').apply(
    lambda x: ';'.join(x), axis=1)
genome_taxa.index.set_names('Feature ID', inplace=True)
genome_taxa.name = "Taxon"
genome_taxa_q2 = Artifact.import_data('FeatureData[Taxonomy]', genome_taxa, pd.Series)

In [10]:
genome_tree_q2 = Artifact.load('data/02.tables/matched_tree_gtdb.qza')

In [11]:
os.makedirs('final-data/02.tables/', exist_ok=True)
species_m_q2.save('final-data/02.tables/matched-species-gtdb.qza')
genome_m_q2.save('final-data/02.tables/matched-genome-gtdb.qza')
genome_taxa_q2.save('final-data/02.tables/matched-taxonomy-gtdb.qza')
genome_tree_q2.save('final-data/02.tables/matched_tree_gtdb.qza')

'final-data/02.tables/matched_tree_gtdb.qza'

We're going to use q2-boots (an update) to do multiple rarefaction and perform diversity calculations. We'll collapse the species level data for alpha diversity and taxonomic beta diversity and do UniFrac on the genome table.

In [12]:
resampled_genome_q2, = q2_boots.resample(genome_m_q2, 
                                         sampling_depth=int(1e6), 
                                         n=25,
                                         replacement=False)
resampled_species_q2 = {id_: q2_taxa.collapse(table_q2, 
                                              taxonomy=genome_taxa_q2, 
                                              level=7).collapsed_table
                        for id_, table_q2 in resampled_genome_q2.items()}

In [13]:
alpha_spec_q2 = {metric: {i: q2_diversity.alpha(table=table_q2, metric=metric).alpha_diversity
                          for i, table_q2 in resampled_species_q2.items()}
                 for metric in ['observed_features', 'shannon', 'simpson']}
alpha_average_q2 = {
    metric: q2_boots.alpha_average(alphas, 
                                   average_method='median').average_alpha_diversity
    for metric, alphas in alpha_spec_q2.items()
    }

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly ins

In [14]:
os.makedirs('final-data/03.alpha-diveristy/', exist_ok=True)
for metric, alpha_q2 in alpha_average_q2.items():
    alpha_q2.save(f'final-data/03.alpha-diveristy/{metric}_1M_avg.qza')

In [15]:
phylo_metrics = ['unweighted_unifrac', 'weighted_unifrac']
taxon_metrics = ['braycurtis', 'jaccard']
betat_spec_q2 = {
    metric: {i: q2_diversity.beta(table=table_q2, metric=metric).distance_matrix
             for i, table_q2 in resampled_species_q2.items()}
    for metric in taxon_metrics
}
betap_spec_q2 = {
    metric: {i: q2_diversity.beta_phylogenetic(table=table_q2, 
                                               phylogeny=genome_tree_q2, 
                                               metric=metric).distance_matrix
             for i, table_q2 in resampled_genome_q2.items()}
    for metric in phylo_metrics
}
betat_spec_q2.update(betap_spec_q2)
beta_average_q2 = {
    metric: q2_boots.beta_average(dms_q2, 
                                  average_method='non-metric-median').average_distance_matrix
    for metric, dms_q2 in betat_spec_q2.items()
    }

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

ssu -i /var/folders/kn/xj9fkj9d48b4n8kr507yzwl00000gn/T/qiime2/jwd/data/d82c1a62-350c-4278-8490-e637c9d09e0e/data/feature-table.biom -t /var/folders/kn/xj9fkj9d48b4n8kr507yzwl00000gn/T/qiime2/jwd/data/f2125f7c-e06e-4227-ba51-a4fa9057fd00/data/tree.nwk -m unweighted -o /var/folders/kn/xj9fkj9d48b4n8kr507yzwl00000gn/T/qiime2/jwd/processes/34346-1764181209.220011@jwd/tmp/q2-OutPath-ho_856fs

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

ssu -i /var/folders/kn/xj9fkj9d48b4n8kr507yzwl00000gn/T/qiime2/jwd/data/81fe3f25-5ccb-4f03-969a-8a2a26b33501/data/feature-table.biom -t /var/f

In [16]:
os.makedirs('final-data/04.beta-diveristy/', exist_ok=True)
for metric, dm_q2 in beta_average_q2.items():
    dm_q2.save(f'final-data/04.beta-diveristy/{metric}_1M_avg_dm.qza')

We'll look at the longitudinal change in alpha diveristy from baseline

In [17]:
alpha_average_q2['simpson'].view(pd.Series).name

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')


'functools.partial(<function _simpsons_dominance at 0x1a279cb80>)'

In [18]:
renames = {
    'shannon': 'shannon_entropy',
    'simpson': alpha_average_q2['simpson'].view(pd.Series).name,
}
alpha_baseline_diffs = {
    metric: q2_longitudinal.first_differences(
        working_meta_q2.merge(art.view(Metadata)),
        metric=renames.get(metric, metric),
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_differences
    for metric, art in alpha_average_q2.items()
}
alpha_baseline_diffs['depth'] = \
    q2_longitudinal.first_differences(
        working_meta_q2,
        metric='gtdb_depth',
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_differences

for metric, alpha in alpha_baseline_diffs.items():
    alpha.save(f'final-data/03.alpha-diveristy/{metric}_1M_avg_from_baseline.qza')

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metad

And we'll do the same with beta diversity.

In [19]:
from_baseline = {
    metric: q2_longitudinal.first_distances(
        distance_matrix=dm_q2,
        metadata=working_meta_q2,
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_distances
    for metric, dm in beta_average_q2.items()
}
for metric, dm_q2 in from_baseline.items():
    dm_q2.save(f'final-data/04.beta-diveristy/{metric}_1M_avg_from_baseline.qza')

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(lambda x: pd.to_numeric(x, errors='ignore'))
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(lambda x: pd.to_numeric(x, errors='ignore'))
/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(lambda x: 

We're adding DEICODE and PhyloDEICODE because they're awesome.

To get there, we'll start by filtering the feature table. Our filtering critiera are that we want features present withh a relative abundance of at least $1\times10^5$ in at least 2 samples in at least 10% of participants.

In [20]:
keep_filt = genome_m_q2.view(biom.Table)
keep_filt.add_metadata(meta.to_dict(orient='index'), axis='sample')
keep_filt.norm(axis='sample', inplace=True)
keep_filt.transform(lambda v, id_, md: v > 5e-5, axis='sample')
keep_filt = keep_filt.collapse(lambda id_, md: md['host_subject_id'],
                               norm=False,
                               axis='sample')
keep_filt.filter(lambda v, id_, md: (v > 2).mean() >= 0.1, 
                 axis='observation')
keep_filt = keep_filt.to_dataframe().sparse.to_dense()
keep_filt.index.set_names('id', inplace=True)

And then we'll filter the features

In [21]:
table_filt_q2, = q2_feature_table.filter_features(table=genome_m_q2,
                                                  metadata=Metadata(keep_filt)
                                                  )
os.makedirs('final-data/05.rpca', exist_ok=True)
table_filt_q2.save('final-data/05.rpca/filtered-table.qza')

'final-data/05.rpca/filtered-table.qza'

I wnat to filter the data to the baseline to calculate the rPCA.

In [22]:
table_filt_bl_q2, = q2_feature_table.filter_samples(table=table_filt_q2,
                                                   metadata=working_meta_q2,
                                                   where='[timepoint] = 0')
deicode_bl_biplot_q2, deicdoe_bl_dm_q2 = \
    q2_gemelli.rpca(table_filt_bl_q2, n_components=3)
phylo_rpca_bl_results = \
    q2_gemelli.phylogenetic_rpca_with_taxonomy(table=table_filt_bl_q2,
                                                  phylogeny=genome_tree_q2,
                                                  taxonomy=genome_taxa_q2.view(Metadata),
                                                  n_components=3
                                                 )

In [23]:
os.makedirs('final-data/05.rpca/baseline_rpca', exist_ok=True)
deicode_bl_biplot_q2.save('final-data/05.rpca/baseline_rpca/biplot.qza')
deicdoe_bl_dm_q2.save('final-data/05.rpca/baseline_rpca/distance-matrix.qza')

os.makedirs('final-data/05.rpca/baseline_phylo_rpca', exist_ok=True)
phylo_rpca_bl_results.biplot.save('final-data/05.rpca/baseline_phylo_rpca/biplot.qza')
phylo_rpca_bl_results.distance_matrix.save('final-data/05.rpca/baseline_phylo_rpca/distance-matrix.qza')
phylo_rpca_bl_results.counts_by_node_tree.save('final-data/05.rpca/baseline_phylo_rpca/counts_by_node_tree.qza')
phylo_rpca_bl_results.counts_by_node.save('final-data/05.rpca/baseline_phylo_rpca/counts_by_node.qza')
phylo_rpca_bl_results.t2t_taxonomy.save('final-data/05.rpca/baseline_phylo_rpca/t2t_taxonomy.qza')

'final-data/05.rpca/baseline_phylo_rpca/t2t_taxonomy.qza'

And then we'll do the rPCA for the full taxonomy.

In [24]:
deicode_results = \
    q2_gemelli.rpca(table_filt_q2, n_components=3)
phylo_rpca_results = \
    q2_gemelli.phylogenetic_rpca_with_taxonomy(table=table_filt_q2,
                                                  phylogeny=genome_tree_q2,
                                                  taxonomy=genome_taxa_q2.view(Metadata),
                                                  n_components=3
                                                 )

In [25]:
os.makedirs('final-data/05.rpca/full_rpca', exist_ok=True)
deicode_results.biplot.save('final-data/05.rpca/full_rpca/biplot.qza')
deicode_results.distance_matrix.save('final-data/05.rpca/full_rpca/distance-matrix.qza')

os.makedirs('final-data/05.rpca/phylo_rpca', exist_ok=True)
phylo_rpca_results.biplot.save('final-data/05.rpca/phylo_rpca/biplot.qza')
phylo_rpca_results.distance_matrix.save('final-data/05.rpca/phylo_rpca/distance-matrix.qza')
phylo_rpca_results.counts_by_node_tree.save('final-data/05.rpca/phylo_rpca/counts_by_node_tree.qza')
phylo_rpca_results.counts_by_node.save('final-data/05.rpca/phylo_rpca/counts_by_node.qza')
phylo_rpca_results.t2t_taxonomy.save('final-data/05.rpca/phylo_rpca/t2t_taxonomy.qza')

'final-data/05.rpca/phylo_rpca/t2t_taxonomy.qza'

# Antibiotic resistance 